# Case study: ưu tiên giữ chân khách hàng

## Mục tiêu
- Đi hết một vòng problem → data → model → recommendation.
- Tách prediction khỏi quyết định kinh doanh.
- Ghi rõ limitation trước khi bàn giao kết quả.

**Business question:** đội retention chỉ liên hệ được 20% khách hàng. Ai nên được ưu tiên nếu mục tiêu là giảm churn trong 30 ngày tới?

## 1. Define decision và metric
- **Unit:** một khách hàng active tại prediction time.
- **Target:** churn trong 30 ngày sau snapshot.
- **Action:** chọn top 20% predicted-risk để nhận offer.
- **Primary metric:** recall/lift trong top 20%, sau đó kiểm tra cost, capacity và fairness.

Dataset bên dưới là synthetic để minh hoạ workflow, không phản ánh khách hàng thật.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X, y = make_classification(
    n_samples=1_000, n_features=6, n_informative=4, weights=[.72, .28], random_state=42
)
features = pd.DataFrame(X, columns=['tenure', 'visits_30d', 'support_tickets', 'avg_order', 'discount_use', 'recency'])
target = pd.Series(y, name='churn')
features.head()

## 2. EDA and data contract
Trước model thật, kiểm tra row uniqueness, feature availability at prediction time, missingness, drift so với kỳ trước và hidden selection bias. Không dùng `cancellation_date` để dự đoán churn: đó là leakage vì chỉ biết sau khi churn xảy ra.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=.25, stratify=target, random_state=42
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
model.fit(X_train, y_train)
risk = model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, risk)

In [ ]:
# Evaluate the actual operational slice: top 20% capacity.
results = X_test.assign(actual=y_test, risk=risk).sort_values('risk', ascending=False)
capacity = max(1, int(len(results) * .20))
top_segment = results.head(capacity)
top_recall = top_segment['actual'].sum() / y_test.sum()
top_churn_rate = top_segment['actual'].mean()
{'capacity': capacity, 'top_20pct_recall': top_recall, 'top_20pct_churn_rate': top_churn_rate}

## 3. Recommendation
Nếu top-20% lift đủ lớn so với random targeting và offer cost thấp hơn expected retained value, hãy chạy controlled experiment: randomize eligible high-risk customers thành treatment/control, theo dõi retention và guardrails (complaint, unsubscribe, margin). Model xếp hạng risk, **không** chứng minh offer sẽ gây retention.

## 4. Limitation
Synthetic data không có temporal shift, intervention history, customer consent hay nhóm bảo vệ. Production cần time-based validation, monitoring, access control, model version, human/business review và thí nghiệm incremental impact.

## Tự luyện
1. Thêm baseline random targeting và tính lift cho top 20%.
2. Chọn một proxy bias có thể xuất hiện trong feature churn; đề xuất cách audit.
3. Viết README một trang cho project thật: context, data, method, result, limitation, next action.

**Checkpoint:** Bạn phân biệt prediction quality với causal impact của chiến dịch.